In [128]:
import math

class Value:
    def __init__(self, data, child=(), op='', label=''):
        self.data = data
        self.prev = set(child)
        self.op = op
        self.grad = 0.0
        self._back_prop = lambda: None
        self.label = label

    def back_prop(self):

        topo = []
        visited = set()
        def build_topo(node):
            if node not in visited:
                visited.add(node)
            for child in node.prev:
                build_topo(child)
            topo.append(node)
        
        build_topo(self)

        self.grad = 1.0
        for val in reversed(topo):
            print(val)
            val._back_prop()
    
    def __repr__(self):
        return f"Value(data = {self.data})"
    
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")
        
        def add_back_prop():
            self.grad = out.grad
            other.grad = out.grad
        out._back_prop = add_back_prop
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')

        def mul_back_prop():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._back_prop = mul_back_prop

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(x * 2) - 1) / (math.exp(x * 2) + 1)
        out = Value(t, (self, ), label='tanh')

        def tanh_back_prop():
            self.grad = (1 - t**2) * out.grad

        out._back_prop = tanh_back_prop
        return out

In [129]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [130]:
x1.grad

0.0

In [131]:
o.back_prop()

Value(data = 0.7071067811865476)
Value(data = 0.8813735870195432)
Value(data = -6.0)
Value(data = 0.0)
Value(data = 1.0)
Value(data = 0.0)
Value(data = -6.0)
Value(data = -3.0)
Value(data = 2.0)
Value(data = 6.881373587019543)


In [132]:
x1.grad

-1.4999999999999996

In [34]:
a = Value(2.0, label="a")
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b
e.label = 'e'
d = e + c
d.label = 'd'
f = Value(-2.0, label='f')
L = d * f

In [37]:
x1 = Value(2.0, label="a")
x2 = Value(-3.0, label='b')
y = x1 * x2
x2.grad

2.0

In [30]:
a

Value(data = 2.0)

In [18]:
def test():
    h = 0.000001
    a = Value(2.0, label="a")
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b
    e.label = 'e'
    d = e + c
    d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f
    L1 = L.data

    a = Value(2.0 + h, label="a")
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b
    e.label = 'e'
    d = e + c
    d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f
    L2 = L.data

    return (L2-L1)/h

test()

6.000000000838668